In [ ]:
"""Actual moleculenet task."""

from fundl.datasets import make_graph_counting_dataset
from fundl.utils import pad_graph
import numpy as onp
import networkx as nx
import jax.numpy as np

import panel as pn
import param
import holoviews as hv
import hvplot
import hvplot.streamz

# hv.extension("bokeh")

In [ ]:
from streamz.dataframe import DataFrame
from streamz import Stream

In [ ]:
# pn.extension()

In [ ]:
from chemgraph import atom_graph, bond_graph
import janitor.chemistry

# Gs = make_graph_counting_dataset(n_graphs=1000)
import pandas as pd

df = (
    pd.read_csv("bace.csv")
    .rename_column("mol", "structure")
    .smiles2mol("structure", "mol")
    .join_apply(lambda x: atom_graph(x["mol"]), "atom_graph")
    .join_apply(lambda x: bond_graph(x["mol"]), "bond_graph")
    .join_apply(lambda x: len(x["atom_graph"]), "atom_graph_size")
    .join_apply(lambda x: len(x["bond_graph"]), "bond_graph_size")
    .shuffle()
)

In [ ]:
def generate_adjacency_and_feature_matrices(Gs):
    Fs = []
    As = []
    for G in Gs:
        Fs.append(np.vstack([d["features"] for n, d in G.nodes(data=True)]))
        As.append(onp.asarray(nx.adjacency_matrix(G).todense()))
        
    return As, Fs

In [ ]:
aGs = df["atom_graph"].tolist()
bGs = df["bond_graph"].tolist()

print("Generating atom graph feature matrices and adjacency matrices...")
aAs, aFs = generate_adjacency_and_feature_matrices(aGs)

In [ ]:
print("Generating bond graph feature matrices and adjacency matrices...")
bAs, bFs = generate_adjacency_and_feature_matrices(bGs)

In [ ]:
largest_agraph_size = max([len(G) for G in aGs])
largest_bgraph_size = max([len(G) for G in bGs])
largest_agraph_size, largest_bgraph_size

In [ ]:
print("Preparing outputs...")
y = df["pIC50"].values.reshape(-1, 1)

y_train = y[0:800, :]
y_test = y[800:, :]

# y_train = onp.ma.masked_array(y_train, value=7.09691)

In [ ]:
def pad_and_stack_graphs(Fs, As, largest_graph_size):
    for i, (F, A) in enumerate(zip(Fs, As)):
        F, A = pad_graph(F, A, largest_graph_size)
        Fs[i] = F
        As[i] = A

    Fs = np.stack(Fs).astype(np.float32)
    As = np.stack(As).astype(np.float32)
    return np.array(Fs), np.array(As)

def simple_split(Fs, As):
    Fs_train, Fs_test = Fs[0:800, :], Fs[800:]
    As_train, As_test = As[0:800, :], As[800:]    
    return Fs_train, Fs_test, As_train, As_test

In [ ]:
# print(aFs.shape), print(aAs.shape)
# print(bFs.shape), print(bAs.shape)

In [ ]:
print("Padding atom graphs to correct size...")
aFs, aAs = pad_and_stack_graphs(aFs, aAs, largest_agraph_size)
bFs, bAs = pad_and_stack_graphs(bFs, bAs, largest_bgraph_size)

aFs_train, aFs_test, aAs_train, aAs_test = simple_split(aFs, aAs)
bFs_train, bFs_test, bAs_train, bAs_test = simple_split(bFs, bAs)


def train_data():
    return aFs_train, aAs_train, bFs_train, bAs_train, y_train

def test_data():
    return aFs_test, aAs_test, bFs_test, bAs_test, y_test

In [ ]:
from fundl.weights import add_dense_params

params = dict()

params = add_dense_params(params, name="agraph1", input_dim=9, output_dim=20)
params = add_dense_params(params, name="agraph2", input_dim=20, output_dim=10)

params = add_dense_params(params, name="bgraph1", input_dim=4, output_dim=20)
params = add_dense_params(params, name="bgraph2", input_dim=20, output_dim=10)

params = add_dense_params(params, name="dense1", input_dim=20, output_dim=1)

In [ ]:
from fundl.layers.graph import mpnn, gather
from fundl.layers import dense
# from fundl.activations import relu
arr = np.array([-1, 1, 0])
def relu(x, alpha=0.1):
    return x * (x > 0) + alpha * x * (x <= 0)

from fundl.losses import _mse_loss
from jax import grad

def mseloss(p, model, aFs, aAs, bFs, bAs, y):
    yhat = model(p, aFs, aAs, bFs, bAs)
    return np.mean(_mse_loss(y, yhat))


def model(params, aFs, aAs, bFs, bAs):
    aFs = mpnn(params["agraph1"], aAs, aFs, nonlin=relu)
    aFs = mpnn(params["agraph2"], aAs, aFs, nonlin=relu)

    bFs = mpnn(params["bgraph1"], bAs, bFs, nonlin=relu)
    bFs = mpnn(params["bgraph2"], bAs, bFs, nonlin=relu)
    
    aOut = gather(aFs)
    bOut = gather(bFs)
    
    out = np.concatenate([aOut, bOut], axis=1)
    out = dense(params["dense1"], out, nonlin=relu)
    return out


dloss = grad(mseloss)

In [ ]:
# As a sanity check, calculate mseloss

out = model(params, aFs, aAs, bFs, bAs)
out.sum()

In [ ]:
from jax.experimental.optimizers import adam

init, update, get_params = adam(step_size=0.001)

state = init(params)
# Make sure things fit over 10 epochs
for i in range(10):
    g = dloss(params, model, aFs_train, aAs_train, bFs_train, bAs_train, y_train)
    state = update(i, g, state)
    params = get_params(state)

    l_test = mseloss(params, model, aFs_test, aAs_test, bFs_test, bAs_test, y_test)
    l_train = mseloss(params, model, aFs_train, aAs_train, bFs_train, bAs_train, y_train)

sum_loss = l_test + l_train + abs(l_test - l_train)
print(sum_loss)

In [ ]:
def fit(n_epochs, model, params, train_data, test_data):
    init, update, get_params = adam(step_size=0.001)
    
    aFs_train, aAs_train, bFs_train, bAs_train, y_train = train_data()
    aFs_test, aAs_test, bFs_test, bAs_test, y_test = test_data()

    state = init(params)
    # Make sure things fit over 10 epochs
    for i in range(n_epochs):
        g = dloss(params, model, aFs_train, aAs_train, bFs_train, bAs_train, y_train)
        state = update(i, g, state)
        params = get_params(state)

        l_test = mseloss(params, model, aFs_test, aAs_test, bFs_test, bAs_test, y_test)
        l_train = mseloss(params, model, aFs_train, aAs_train, bFs_train, bAs_train, y_train)
        if i % 10 == 0:
            print(l_train, l_test)
    return l_train, l_test, params

In [ ]:
def objective(trial):
    n_epochs = trial.suggest_int("n_epochs", 30, 1000)
    loss_train, loss_test, params = fit(n_epochs, model, params, train_data, test_data)
    return abs(loss_train - loss_test) + loss_train + loss_test

In [ ]:
import optuna

study = optuna.create_study()
study.optimize(objective, n_trials=20)

In [ ]:
study.best_params

In [ ]:
loss_train, loss_test, params = fit(study.best_params["n_epochs"], model, params, train_data, test_data)

In [ ]:
loss_train, loss_test

In [ ]:
preds_train = model(params, aFs_train, aAs_train, bFs_train, bAs_train)
preds_test = model(params, aFs_test, aAs_test, bFs_test, bAs_test)
# hv.Scatter((preds_train, y_train)) * hv.Scatter((preds_test, y_test)) * hv.Curve(data=[(0, 0), (20, 20)])

In [ ]:
y_train.squeeze().shape

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
import matplotlib.pyplot as plt


plt.scatter(y_train.squeeze(), preds_train.squeeze())
plt.scatter(y_test.squeeze(), preds_test.squeeze())